In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/workspace/ProtoSAM-repro")
BASE_ESPERADA = "64868c909668aac9ad517f7192a2712e88268e8f"
PYTHON_ESPERADO = Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
)

# Confere que o kernel usa o ambiente esperado.
assert Path(sys.executable).absolute() == PYTHON_ESPERADO, (
    f"Kernel incorreto: {sys.executable}"
)

# Um kernel novo não deve ter módulos do projeto já carregados.
carregados = [
    nome for nome in sys.modules
    if nome in ("models", "dataloaders", "util")
    or nome.startswith(("models.", "dataloaders.", "util."))
]
assert not carregados, (
    f"Há módulos do projeto carregados: {carregados[:10]}. "
    "Reinicie o kernel antes de continuar."
)

os.chdir(REPO)
sys.path.insert(0, str(REPO))

def git(*args):
    return subprocess.check_output(
        ["git", *args], cwd=REPO, text=True
    ).strip()

assert git("rev-parse", "HEAD") == BASE_ESPERADA
assert git("branch", "--show-current") == "repro/vmamba-clean"
assert not git("diff", "HEAD", "--name-only"), (
    "Há alterações em arquivos rastreados."
)

import torch

print("Python:", sys.executable)
print("Diretório:", Path.cwd())
print("Branch:", git("branch", "--show-current"))
print("Commit:", git("rev-parse", "HEAD"))
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("\nKernel preparado para a base oficial.")

In [ ]:
import json
from pathlib import Path

REPO_ANTIGO = Path("/workspace/ProtoSAM")
REPO_NOVO = Path("/workspace/ProtoSAM-repro")

vinculos = [
    (
        REPO_ANTIGO / "pretrained_model",
        REPO_NOVO / "pretrained_model",
    ),
    (
        REPO_ANTIGO / "data/PolypDataset",
        REPO_NOVO / "data/PolypDataset",
    ),
]

# Confere todos os caminhos antes de criar qualquer vínculo.
for origem, destino in vinculos:
    assert origem.is_dir(), f"Origem não encontrada: {origem}"

    if destino.is_symlink():
        assert destino.resolve() == origem.resolve(), (
            f"Já existe um link para outra origem: {destino}"
        )
    else:
        assert not destino.exists(), (
            f"O destino já existe e não será substituído: {destino}"
        )

for origem, destino in vinculos:
    if not destino.is_symlink():
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.symlink_to(origem, target_is_directory=True)

# Verifica os pesos conhecidos, sem carregar modelos.
pesos = {
    "DINOv2-L": (
        REPO_NOVO
        / "pretrained_model/hub/checkpoints/dinov2_vitl14_pretrain.pth"
    ),
    "SAM-H": REPO_NOVO / "pretrained_model/sam_vit_h.pth",
}

registro = {
    "vinculos": [
        {"destino": str(destino), "origem": str(destino.resolve())}
        for origem, destino in vinculos
    ],
    "pesos": {
        nome: {
            "existe": caminho.is_file(),
            "tamanho_GiB": (
                round(caminho.stat().st_size / 1024**3, 3)
                if caminho.is_file() else None
            ),
        }
        for nome, caminho in pesos.items()
    },
    "codigo_dinov2_local_existe": (
        REPO_NOVO / "pretrained_model/hub/facebookresearch_dinov2_main"
    ).is_dir(),
}

print(json.dumps(registro, indent=2, ensure_ascii=False))

In [ ]:
import ast
import json
import sys
import subprocess
import torch
from datetime import datetime, timezone

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = REPO_NOVO / "reproducao" / f"00_baseline_original_{identificador}"
SESSAO_DIR = RUN_DIR / "sessoes" / identificador
SESSAO_DIR.mkdir(parents=True, exist_ok=False)

def salvar_json(caminho, conteudo):
    with caminho.open("x", encoding="utf-8") as arquivo:
        json.dump(conteudo, arquivo, indent=2, ensure_ascii=False)

ambiente = {
    "repositorio": str(REPO_NOVO),
    "commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_NOVO, text=True
    ).strip(),
    "python": sys.version,
    "executavel": sys.executable,
    "pytorch": torch.__version__,
    "cuda_pytorch": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "threads": torch.get_num_threads(),
    "threads_interop": torch.get_num_interop_threads(),
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "pendente": ["inventario completo de dependencias, driver e CPU"],
}
salvar_json(SESSAO_DIR / "ambiente.json", ambiente)
salvar_json(RUN_DIR / "recursos_compartilhados.json", registro)

with (RUN_DIR / "pergunta.md").open("x", encoding="utf-8") as arquivo:
    arquivo.write(
        "# Baseline oficial\n\n"
        "Conseguimos executar e compreender o pipeline oficial "
        "DINOv2-L + ALP + SAM-H usando pesos locais?\n\n"
        "Primeiro validar um par de suporte/query e seus intermediários. "
        "Depois definir a avaliação completa.\n\n"
        "Ainda sem predições, métricas ou medições de tempo. "
        "Configuração efetiva e demais registros pendentes.\n"
    )

caminho = REPO_NOVO / "models/grid_proto_fewshot.py"
fonte = caminho.read_text(encoding="utf-8")
arvore = ast.parse(fonte)
trechos = []

for classe in arvore.body:
    if isinstance(classe, ast.ClassDef) and classe.name == "FewShotSeg":
        for metodo in classe.body:
            if isinstance(metodo, ast.FunctionDef) and metodo.name in (
                "__init__", "get_encoder"
            ):
                trechos.append(ast.get_source_segment(fonte, metodo))

assert trechos, "Métodos não encontrados; precisamos conferir o arquivo."
texto = "\n\n".join(trechos)

with (SESSAO_DIR / "construcao_encoder_oficial.txt").open(
    "x", encoding="utf-8"
) as arquivo:
    arquivo.write(texto)

print(texto)
print(f"\nExecução: {RUN_DIR}")
print(f"Sessão: {SESSAO_DIR}")

In [ ]:
import json
import torch
import models.grid_proto_fewshot as modulo_alp
from datetime import datetime, timezone

# Confirma que o módulo importado pertence ao worktree novo.
assert REPO_NOVO.resolve() in Path(modulo_alp.__file__).resolve().parents, (
    f"Módulo carregado de outro diretório: {modulo_alp.__file__}"
)

DINO_REPO = REPO_NOVO / "pretrained_model/hub/facebookresearch_dinov2_main"
DINO_PESOS = (
    REPO_NOVO
    / "pretrained_model/hub/checkpoints/dinov2_vitl14_pretrain.pth"
)

assert DINO_REPO.is_dir()
assert DINO_PESOS.is_file()

class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(DINO_REPO),
            "dinov2_vitl14",
            source="local",
            pretrained=False,
        )
        pesos = torch.load(
            DINO_PESOS,
            map_location="cpu",
            weights_only=True,
        )
        resultado = self.encoder.load_state_dict(pesos, strict=True)
        print("Carregamento dos pesos:", resultado)
        del pesos

        # Mesma regra da implementação oficial.
        lado = max(
            self.image_size // 14,
            modulo_alp.DEFAULT_FEATURE_SIZE,
        )
        self.config["feature_hw"] = [lado, lado]

config_alp = {
    "which_model": "dinov2_l14",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False,
}

alpnet = FewShotSegDinoLocal(
    image_size=672,
    pretrained_path=None,
    cfg=config_alp.copy(),
).to(device="cuda", dtype=torch.float32)

alpnet.eval()
alpnet.requires_grad_(False)

registro_modelo = {
    "modulo_fewshotseg": str(Path(modulo_alp.__file__).resolve()),
    "adaptacao": "Somente get_encoder: codigo e pesos DINOv2 locais",
    "codigo_dinov2": str(DINO_REPO.resolve()),
    "pesos_dinov2": str(DINO_PESOS.resolve()),
    "image_size": alpnet.image_size,
    "config_efetiva": alpnet.config,
    "kernel_size_alp": alpnet.cls_unit.kernel_size,
    "dtype": str(next(alpnet.encoder.parameters()).dtype),
    "device": str(next(alpnet.encoder.parameters()).device),
    "modo_treino": alpnet.training,
    "parametros_treinaveis": sum(
        p.numel() for p in alpnet.parameters() if p.requires_grad
    ),
    "sam": "Ainda nao carregado",
}

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"carregamento_dinov2_{identificador}.json"
with destino.open("x", encoding="utf-8") as arquivo:
    json.dump(registro_modelo, arquivo, indent=2, ensure_ascii=False)

print(json.dumps(registro_modelo, indent=2, ensure_ascii=False))
print(f"\nSalvo: {destino}")

In [ ]:
import ast
from datetime import datetime, timezone

alvos = {
    "dataloaders/PolypDataset.py": {
        "PolypDataset": {"__init__", "process_image_gt"},
    },
    "models/segment_anything/utils/transforms.py": {
        "ResizeLongestSide": {
            "__init__", "apply_image_torch", "preprocess",
        },
    },
}

partes = []

for relativo, classes in alvos.items():
    fonte = (REPO_NOVO / relativo).read_text(encoding="utf-8")
    arvore = ast.parse(fonte)

    for classe in arvore.body:
        if not isinstance(classe, ast.ClassDef):
            continue
        if classe.name not in classes:
            continue

        for metodo in classe.body:
            if (
                isinstance(metodo, ast.FunctionDef)
                and metodo.name in classes[classe.name]
            ):
                partes.append(
                    f"\n--- {relativo}: {classe.name}.{metodo.name} ---"
                )
                partes.append(ast.get_source_segment(fonte, metodo))

# Identifica como a avaliação oficial conecta o resize ao dataset.
relativo = "validation_protosam.py"
linhas = (REPO_NOVO / relativo).read_text(encoding="utf-8").splitlines()
selecionadas = set()

for i, linha in enumerate(linhas):
    if any(termo in linha for termo in (
        "get_polyp_dataset(", "ResizeLongestSide(", "sam_trans="
    )):
        selecionadas.update(
            range(max(0, i - 5), min(len(linhas), i + 6))
        )

partes.append(f"\n--- {relativo}: preparação do dataset ---")
anterior = -2
for i in sorted(selecionadas):
    if i > anterior + 1:
        partes.append("...")
    partes.append(f"{i + 1:04d}: {linhas[i]}")
    anterior = i

texto = "\n".join(partes)
print(texto)

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"preprocessamento_oficial_{identificador}.txt"
with destino.open("x", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print(f"\nSalvo: {destino}")

In [ ]:
import json
import torch
from pathlib import Path
from datetime import datetime, timezone
from dataloaders.PolypDataset import get_polyp_dataset
from models.segment_anything.utils.transforms import ResizeLongestSide

sam_trans = ResizeLongestSide(1024)

print("Inicializando os datasets oficiais...", flush=True)
ds_suportes, ds_teste = get_polyp_dataset(
    image_size=(1024, 1024),
    sam_trans=sam_trans,
)

nomes_par = {
    "suporte": "cju3xtufwiv9c0818djsc4cqd.png",
    "query": "cju85nr8elly209872w9n5m0s.png",
}

entradas_oficiais = {}
registro_par = {
    "escopo": "Par de inspecao; query fora do conjunto de teste",
    "preprocessamento": "PolypDataset oficial, sem alteracoes",
    "exemplos": {},
}

for papel, nome in nomes_par.items():
    indices = [
        i for i, caminho in enumerate(ds_suportes.images)
        if Path(caminho).name == nome
    ]
    assert len(indices) == 1, f"Esperado um único arquivo: {nome}"
    indice = indices[0]

    caminho_imagem = ds_suportes.images[indice]
    caminho_mascara = ds_suportes.gts[indice]
    assert Path(caminho_imagem).stem == Path(caminho_mascara).stem

    # Usa o loader oficial para conferir dimensões antes da transformação.
    original = ds_suportes.cv2_loader(caminho_imagem, is_mask=False)
    h, w = map(int, original.shape[:2])

    # Executa __getitem__ e process_image_gt oficiais.
    exemplo = ds_suportes[indice]
    imagem = exemplo["image"].detach().cpu()
    mascara = exemplo["label"].detach().cpu()

    assert imagem.shape == (3, 1024, 1024)
    assert mascara.shape == (1024, 1024)
    assert torch.isfinite(imagem).all()
    assert ((mascara == 0) | (mascara == 1)).all()

    entradas_oficiais[papel] = {
        "imagem": imagem,
        "mascara": mascara,
        "tamanho_original": (h, w),
        "tamanho_redimensionado": sam_trans.get_preprocess_shape(h, w, 1024),
        "original_size_retornado": exemplo["original_size"],
    }

    registro_par["exemplos"][papel] = {
        "nome": nome,
        "indice": indice,
        "imagem": str(caminho_imagem),
        "mascara": str(caminho_mascara),
        "dtype_loader": str(original.dtype),
        "tamanho_original_real": [h, w],
        "original_size_retornado": exemplo["original_size"].tolist(),
        "tamanho_redimensionado":
            list(entradas_oficiais[papel]["tamanho_redimensionado"]),
        "shape_imagem": list(imagem.shape),
        "shape_mascara": list(mascara.shape),
        "pixels_objeto": int(mascara.sum().item()),
    }

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PREPARO_DIR = SESSAO_DIR / f"preparo_par_{identificador}"
PREPARO_DIR.mkdir(exist_ok=False)

torch.save(entradas_oficiais, PREPARO_DIR / "entradas_oficiais.pt")
with (PREPARO_DIR / "par_selecionado.json").open(
    "x", encoding="utf-8"
) as arquivo:
    json.dump(registro_par, arquivo, indent=2, ensure_ascii=False)

print(json.dumps(registro_par, indent=2, ensure_ascii=False))
print(f"\nSalvo: {PREPARO_DIR}")

In [ ]:
import csv
import json
import torch
from datetime import datetime, timezone

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PRED_ALP_DIR = SESSAO_DIR / f"predicao_alp_{identificador}"
PRED_ALP_DIR.mkdir(exist_ok=False)

config_predicao = {
    "escopo": "FewShotSeg oficial direto; sem wrapper ProtoSAM e sem SAM",
    "image_size_encoder": 672,
    "isval": True,
    "val_wsize": 2,
    "show_viz": False,
    "supp_fts": None,
    "preparo": str(PREPARO_DIR),
    "avaliacao": "Mascara oficial pre-processada, sem padding",
    "tempos": "Nao medidos",
}
with (PRED_ALP_DIR / "config.json").open("x", encoding="utf-8") as arquivo:
    json.dump(config_predicao, arquivo, indent=2, ensure_ascii=False)

suporte = entradas_oficiais["suporte"]
query = entradas_oficiais["query"]
dispositivo = next(alpnet.parameters()).device

imagem_s = suporte["imagem"].unsqueeze(0).to(dispositivo)
imagem_q = query["imagem"].unsqueeze(0).to(dispositivo)
mascara_s = suporte["mascara"].unsqueeze(0).to(dispositivo)

alpnet.eval()
with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    retorno = alpnet(
        supp_imgs=[[imagem_s]],
        fore_mask=[[mascara_s]],
        back_mask=[[1.0 - mascara_s]],
        qry_imgs=[imagem_q],
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )
    escores_oficiais = retorno[0].detach().cpu()

assert escores_oficiais.shape == (1, 2, 1024, 1024)
assert torch.isfinite(escores_oficiais).all()

predicao_oficial = escores_oficiais.argmax(dim=1)[0].to(torch.uint8)
h, w = map(int, query["tamanho_redimensionado"])
pred = predicao_oficial[:h, :w].bool()
gt = query["mascara"][:h, :w].bool()
assert gt.any()

tp = int((pred & gt).sum().item())
fp = int((pred & ~gt).sum().item())
fn = int((~pred & gt).sum().item())

metricas_oficiais = {
    "query": nomes_par["query"],
    "suporte": nomes_par["suporte"],
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "Dice": 2 * tp / (2 * tp + fp + fn),
    "IoU": tp / (tp + fp + fn),
    "precisao": tp / (tp + fp) if tp + fp else None,
    "recall": tp / (tp + fn),
    "arquivo_predicao": str(PRED_ALP_DIR / "predicao.pt"),
    "arquivo_referencia": str(PREPARO_DIR / "entradas_oficiais.pt"),
}

torch.save(
    {"escores": escores_oficiais, "predicao": predicao_oficial},
    PRED_ALP_DIR / "predicao.pt",
)
with (PRED_ALP_DIR / "por_imagem.csv").open(
    "x", newline="", encoding="utf-8"
) as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=list(metricas_oficiais))
    writer.writeheader()
    writer.writerow(metricas_oficiais)

del retorno
print(json.dumps(metricas_oficiais, indent=2, ensure_ascii=False))
print(f"\nSalvo: {PRED_ALP_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import Image, display

query = entradas_oficiais["query"]
h, w = map(int, query["tamanho_redimensionado"])

media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)

rgb = (
    (query["imagem"] * desvio + media)[:, :h, :w]
    .clamp(0, 255)
    .permute(1, 2, 0)
    .numpy() / 255.0
)

gt = query["mascara"][:h, :w].numpy().astype(bool)
pred = predicao_oficial[:h, :w].numpy().astype(bool)

erros = np.zeros((*gt.shape, 4), dtype=np.float32)
erros[pred & gt] = [0.0, 1.0, 0.0, 0.45]
erros[pred & ~gt] = [1.0, 0.0, 0.0, 0.65]
erros[~pred & gt] = [0.0, 0.4, 1.0, 0.65]

fig, eixos = plt.subplots(1, 3, figsize=(15, 5))

eixos[0].imshow(rgb)
eixos[0].set_title("Query")

eixos[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
eixos[1].set_title("Referência oficial pré-processada")

eixos[2].imshow(rgb)
eixos[2].imshow(erros)
eixos[2].set_title(
    f"ALP oficial — Dice {metricas_oficiais['Dice']:.2%}"
)

for eixo in eixos:
    eixo.axis("off")

fig.legend(
    handles=[
        Patch(facecolor="lime", label="Acerto — TP"),
        Patch(facecolor="red", label="Excesso — FP"),
        Patch(facecolor="dodgerblue", label="Omissão — FN"),
    ],
    loc="lower center",
    ncol=3,
)
fig.tight_layout(rect=[0, 0.08, 1, 1])

pasta = PRED_ALP_DIR / "figuras"
pasta.mkdir(exist_ok=True)
destino = pasta / "alp_oficial_erros.png"
fig.savefig(destino, dpi=150, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(destino)))
print(f"Salvo: {destino}")

In [ ]:
import ast
from datetime import datetime, timezone

caminho = REPO_NOVO / "models/ProtoSAM.py"
fonte = caminho.read_text(encoding="utf-8")
arvore = ast.parse(fonte)
partes = []

# Mostra de onde vêm as funções auxiliares.
partes.append("--- Imports ---")
for no in arvore.body:
    if isinstance(no, (ast.Import, ast.ImportFrom)):
        partes.append(ast.get_source_segment(fonte, no))

for classe in arvore.body:
    if not isinstance(classe, ast.ClassDef):
        continue
    if classe.name not in ("ALPNetWrapper", "ProtoSAM"):
        continue

    metodos = [
        no for no in classe.body
        if isinstance(no, ast.FunctionDef)
    ]
    partes.append(f"\n--- Classe {classe.name} ---")
    partes.append("Métodos: " + ", ".join(m.name for m in metodos))

    for metodo in metodos:
        if classe.name == "ALPNetWrapper" or metodo.name in (
            "__init__", "forward", "__call__", "get_prediction", "predict"
        ):
            partes.append(ast.get_source_segment(fonte, metodo))

texto = "\n\n".join(partes)
print(texto)

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"wrapper_oficial_{identificador}.txt"
with destino.open("x", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print(f"\nSalvo: {destino}")

In [ ]:
import ast
from datetime import datetime, timezone

alvos = {
    "models/ProtoSAM.py": {
        "get_sam",
        "get_bbox",
        "get_bbox_per_cc",
        "get_most_conf_points",
        "get_sam_input_points",
        "predict_w_points_bbox",
    },
    "util/utils.py": {
        "cca",
        "get_connected_components",
        "get_confidence_from_logits",
    },
}

partes = []

for relativo, nomes in alvos.items():
    fonte = (REPO_NOVO / relativo).read_text(encoding="utf-8")
    arvore = ast.parse(fonte)

    for no in ast.walk(arvore):
        if isinstance(no, ast.FunctionDef) and no.name in nomes:
            partes.append(f"\n--- {relativo}: {no.name} ---")
            partes.append(ast.get_source_segment(fonte, no))

# Mostra os parâmetros no config e eventuais sobrescritas no script.
termos = (
    "use_points", "use_bbox", "use_mask", "do_cca",
    "point_mode", "use_neg_points", "coarse_pred_only",
    "protosam_sam_ver", "sam_checkpoint", "val_wsize",
    "CONF_MODE", "POINT_MODES",
)

for relativo in (
    "config_ssl_upload.py",
    "run_protosam.sh",
    "models/ProtoSAM.py",
):
    linhas = (REPO_NOVO / relativo).read_text(
        encoding="utf-8"
    ).splitlines()

    partes.append(f"\n--- Parâmetros e constantes: {relativo} ---")
    for i, linha in enumerate(linhas, start=1):
        if any(termo in linha for termo in termos):
            partes.append(f"{i:04d}: {linha}")

texto = "\n".join(partes)
print(texto)

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"extracao_prompts_oficial_{identificador}.txt"
with destino.open("x", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print(f"\nSalvo: {destino}")

In [ ]:
import json
import numpy as np
from datetime import datetime, timezone
from models.ProtoSAM import ProtoSAM
from util.utils import get_connected_components

# Reutiliza somente os métodos de prompts, sem construir o modelo SAM.
class ExtratorPrompts:
    get_bbox_per_cc = ProtoSAM.get_bbox_per_cc
    get_most_conf_points = ProtoSAM.get_most_conf_points
    get_sam_input_points = ProtoSAM.get_sam_input_points

    num_points_for_sam = 1
    point_mode = "both"
    debug = False

extrator = ExtratorPrompts()
logits = escores_oficiais.detach().cpu()
probabilidades = logits.softmax(dim=1)
mascara_inicial = logits.argmax(dim=1)[0].numpy().astype(np.uint8)

assert mascara_inicial.any(), "Sem primeiro plano para extrair prompts."

componentes, confiancas = get_connected_components(
    mascara_inicial, logits, return_conf=True
)
caixas = extrator.get_bbox_per_cc(componentes)

pontos, rotulos_aux, pontos_neg, rotulos_neg_aux = (
    extrator.get_sam_input_points(
        componentes,
        probabilidades,
        get_neg_points=False,
        l=1,
    )
)

prompts_oficiais = []
for indice, caixa in enumerate(caixas):
    cc_id = indice + 1
    prompts_oficiais.append({
        "componente": cc_id,
        "area_pixels": int((componentes[1] == cc_id).sum()),
        "confianca_ponderada": float(confiancas[cc_id]),
        "bbox_xyxy": np.asarray(caixa).tolist(),
        "pontos_xy": np.asarray(pontos[indice]).tolist(),
        "ordem_pontos": ["maior_confianca", "centroide"],
        # predict_w_points_bbox recria esses rótulos como positivos.
        "rotulos_enviados_ao_sam": [1] * len(pontos[indice]),
    })

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PROMPTS_DIR = PRED_ALP_DIR / f"prompts_{identificador}"
PROMPTS_DIR.mkdir(exist_ok=False)

np.save(PROMPTS_DIR / "componentes.npy", componentes[1])
with (PROMPTS_DIR / "prompts.json").open("x", encoding="utf-8") as arquivo:
    json.dump({
        "escopo": "Extracao em CPU dos escores ALP salvos; SAM nao executado",
        "coordenadas": "Canvas 1024x1024, incluindo padding; pontos em x,y",
        "use_cca": False,
        "point_mode": "both",
        "use_bbox": True,
        "use_neg_points": False,
        "prompts": prompts_oficiais,
    }, arquivo, indent=2, ensure_ascii=False)

print(f"Regiões de primeiro plano: {componentes[0] - 1}")
print(json.dumps(prompts_oficiais, indent=2, ensure_ascii=False))
print(f"\nSalvo: {PROMPTS_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import Image, display

query = entradas_oficiais["query"]
h, w = map(int, query["tamanho_redimensionado"])

media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)
rgb = (
    (query["imagem"] * desvio + media)[:, :h, :w]
    .clamp(0, 255)
    .permute(1, 2, 0)
    .numpy() / 255.0
)

principal = max(
    prompts_oficiais, key=lambda p: p["confianca_ponderada"]
)

fig, eixos = plt.subplots(1, 2, figsize=(16, 7))

for eixo in eixos:
    eixo.imshow(rgb)

for prompt in prompts_oficiais:
    x1, y1, x2, y2 = prompt["bbox_xyxy"]
    pontos_xy = np.asarray(prompt["pontos_xy"])
    cor = "lime" if prompt["componente"] == principal["componente"] else "orange"

    # No painel direito, exibe apenas os prompts do componente principal.
    paineis = eixos if prompt is principal else [eixos[0]]

    for eixo in paineis:
        eixo.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor=cor, linewidth=1.5,
        ))
        eixo.scatter(
            *pontos_xy[0], marker="*", s=130,
            c="cyan", edgecolors="black", linewidths=0.5,
        )
        eixo.scatter(
            *pontos_xy[1], marker="x", s=65,
            c="magenta", linewidths=2,
        )
        eixo.annotate(
            str(prompt["componente"]), (x1, y1),
            xytext=(3, -10), textcoords="offset points",
            color=cor, fontsize=10,
            bbox=dict(facecolor="black", alpha=0.6, pad=1),
        )

x1, y1, x2, y2 = principal["bbox_xyxy"]
margem = 45
eixos[1].set_xlim(max(0, x1 - margem), min(w, x2 + margem))
eixos[1].set_ylim(min(h, y2 + margem), max(0, y1 - margem))

eixos[0].set_title("Todos os 14 conjuntos de prompts")
eixos[1].set_title(f"Detalhe do componente {principal['componente']}")

for eixo in eixos:
    eixo.axis("off")

fig.legend(handles=[
    Line2D([], [], color="cyan", marker="*", linestyle="None",
           markersize=12, label="Maior confiança"),
    Line2D([], [], color="magenta", marker="x", linestyle="None",
           markersize=9, label="Centroide"),
], loc="lower center", ncol=2)

fig.tight_layout(rect=[0, 0.07, 1, 1])
destino = PROMPTS_DIR / "prompts_sobre_query.png"
fig.savefig(destino, dpi=150, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(destino)))
print(f"Salvo: {destino}")

In [ ]:
import inspect
import json
import torch
from datetime import datetime, timezone
from models.ProtoSAM import ProtoSAM, ALPNetWrapper

SAM_PESOS = REPO_NOVO / "pretrained_model/sam_vit_h.pth"
assert SAM_PESOS.is_file()
assert "protosam" not in globals(), (
    "O objeto protosam já existe. Não execute novamente para evitar duplicação."
)

config_protosam = {
    "image_size": [1024, 1024],
    "sam_pretrained_path": str(SAM_PESOS),
    "num_points_for_sam": 1,  # Em modo 'both', soma-se o centroide.
    "use_points": True,
    "use_bbox": True,
    "use_mask": False,
    "debug": False,
    "use_cca": False,
    "point_mode": "both",
    "use_sam_trans": True,
    "coarse_pred_only": False,
    "use_neg_points": False,
}

print("Carregando SAM-H dos pesos locais...", flush=True)

protosam = ProtoSAM(
    coarse_segmentation_model=ALPNetWrapper(alpnet),
    **config_protosam,
)
protosam.to(device="cuda", dtype=torch.float32)
protosam.eval()
protosam.requires_grad_(False)

# Mantém também o encoder dentro do wrapper explicitamente congelado.
alpnet.eval()
alpnet.requires_grad_(False)

assert not protosam.training
assert not protosam.sam.training
assert next(protosam.sam.parameters()).device.type == "cuda"

registro_sam = {
    "config": config_protosam,
    "pesos_resolvidos": str(SAM_PESOS.resolve()),
    "arquivo_classe_sam": inspect.getfile(type(protosam.sam)),
    "arquivo_predictor": inspect.getfile(type(protosam.predictor)),
    "dtype": str(next(protosam.sam.parameters()).dtype),
    "device": str(next(protosam.sam.parameters()).device),
    "parametros_sam": sum(p.numel() for p in protosam.sam.parameters()),
    "parametros_sam_treinaveis": sum(
        p.numel() for p in protosam.sam.parameters() if p.requires_grad
    ),
    "media_transformador_wrapper":
        protosam.sam_trans.pixel_mean.flatten().tolist(),
    "desvio_transformador_wrapper":
        protosam.sam_trans.pixel_std.flatten().tolist(),
    "inferencia": "Ainda nao executada",
}

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"carregamento_sam_{identificador}.json"
with destino.open("x", encoding="utf-8") as arquivo:
    json.dump(registro_sam, arquivo, indent=2, ensure_ascii=False)

print(json.dumps(registro_sam, indent=2, ensure_ascii=False))
print(f"\nSalvo: {destino}")

In [ ]:
import csv
import json
import time
import torch
from datetime import datetime, timezone
from models.ProtoSAM import ALPNetInput

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PRED_SAM_DIR = SESSAO_DIR / f"predicao_protosam_{identificador}"
PRED_SAM_DIR.mkdir(exist_ok=False)

dispositivo = next(alpnet.parameters()).device
suporte = entradas_oficiais["suporte"]
query = entradas_oficiais["query"]

imagem_s = suporte["imagem"].unsqueeze(0).to(dispositivo)
imagem_q = query["imagem"].unsqueeze(0).to(dispositivo)
mascara_s = suporte["mascara"].unsqueeze(0).to(dispositivo)

entrada_alp = ALPNetInput(
    support_images=[imagem_s],
    support_labels=[mascara_s],
    query_images=imagem_q,
    isval=True,
    val_wsize=2,
    show_viz=False,
    supp_fts=None,
)

config_execucao = {
    "protosam": config_protosam,
    "alp": alpnet.config,
    "image_size_encoder": alpnet.image_size,
    "isval": True,
    "val_wsize": 2,
    "degrees_rotate": 0,
    "preparo": str(PREPARO_DIR),
    "avaliacao": "Referencia oficial pre-processada, sem padding",
    "tempo": "Uma chamada sem aquecimento; diagnostico, nao benchmark",
}
with (PRED_SAM_DIR / "config.json").open("x", encoding="utf-8") as arquivo:
    json.dump(config_execucao, arquivo, indent=2, ensure_ascii=False)

print("Executando DINOv2 + ALP + SAM-H...", flush=True)

protosam.eval()
alpnet.eval()
with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    torch.cuda.synchronize()
    inicio = time.perf_counter()

    predicao_sam, scores_sam = protosam(
        query_image=imagem_q,
        coarse_model_input=entrada_alp,
        degrees_rotate=0,
    )

    torch.cuda.synchronize()
    duracao = time.perf_counter() - inicio

predicao_sam = predicao_sam.detach().cpu()
assert predicao_sam.shape == (1024, 1024)
assert torch.isfinite(predicao_sam).all()
assert ((predicao_sam == 0) | (predicao_sam == 1)).all()

torch.save(
    {
        "predicao": predicao_sam.to(torch.uint8),
        "scores_sam": [float(s) for s in scores_sam],
    },
    PRED_SAM_DIR / "predicao.pt",
)

h, w = map(int, query["tamanho_redimensionado"])
pred = predicao_sam[:h, :w].bool()
gt = query["mascara"][:h, :w].bool()
assert gt.any()

tp = int((pred & gt).sum().item())
fp = int((pred & ~gt).sum().item())
fn = int((~pred & gt).sum().item())

metricas_sam = {
    "query": nomes_par["query"],
    "suporte": nomes_par["suporte"],
    "TP": tp, "FP": fp, "FN": fn,
    "Dice": 2 * tp / (2 * tp + fp + fn),
    "IoU": tp / (tp + fp + fn),
    "precisao": tp / (tp + fp) if tp + fp else None,
    "recall": tp / (tp + fn),
    "numero_scores_sam": len(scores_sam),
    "arquivo_predicao": str(PRED_SAM_DIR / "predicao.pt"),
    "arquivo_referencia": str(PREPARO_DIR / "entradas_oficiais.pt"),
}

with (PRED_SAM_DIR / "por_imagem.csv").open(
    "x", newline="", encoding="utf-8"
) as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=list(metricas_sam))
    writer.writeheader()
    writer.writerow(metricas_sam)

with (PRED_SAM_DIR / "tempos.csv").open(
    "x", newline="", encoding="utf-8"
) as arquivo:
    writer = csv.writer(arquivo)
    writer.writerow(["medicao", "segundos", "aquecimento", "escopo"])
    writer.writerow([1, duracao, False, "forward_completo_sem_carregamento"])

print(json.dumps(metricas_sam, indent=2, ensure_ascii=False))
print(f"\nTempo desta chamada: {duracao:.3f} s — não é benchmark")
print(f"Salvo: {PRED_SAM_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import Image, display

query = entradas_oficiais["query"]
h, w = map(int, query["tamanho_redimensionado"])

media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)
rgb = (
    (query["imagem"] * desvio + media)[:, :h, :w]
    .clamp(0, 255).permute(1, 2, 0).numpy() / 255.0
)
gt = query["mascara"][:h, :w].numpy().astype(bool)

comparacoes = [
    ("ALP", predicao_oficial, metricas_oficiais["Dice"]),
    ("ALP + SAM", predicao_sam, metricas_sam["Dice"]),
]

fig, eixos = plt.subplots(1, 3, figsize=(18, 6))
eixos[0].imshow(rgb)
eixos[0].contour(
    gt.astype(float), levels=[0.5], colors=["yellow"], linewidths=1
)
eixos[0].set_title("Query e contorno da referência")

for eixo, (nome, predicao, dice) in zip(eixos[1:], comparacoes):
    pred = predicao[:h, :w].detach().cpu().numpy().astype(bool)

    camada = np.zeros((*gt.shape, 4), dtype=np.float32)
    camada[pred & gt] = [0, 1, 0, 0.45]
    camada[pred & ~gt] = [1, 0, 0, 0.65]
    camada[~pred & gt] = [0, 0.4, 1, 0.65]

    eixo.imshow(rgb)
    eixo.imshow(camada)
    eixo.set_title(f"{nome} — Dice {dice:.2%}")

for eixo in eixos:
    eixo.axis("off")

fig.legend(handles=[
    Patch(facecolor="lime", label="Acerto — TP"),
    Patch(facecolor="red", label="Excesso — FP"),
    Patch(facecolor="dodgerblue", label="Omissão — FN"),
], loc="lower center", ncol=3)

fig.tight_layout(rect=[0, 0.08, 1, 1])
pasta = PRED_SAM_DIR / "figuras"
pasta.mkdir(exist_ok=True)
destino = pasta / "comparacao_alp_sam.png"
fig.savefig(destino, dpi=150, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(destino)))
print(f"Salvo: {destino}")

In [ ]:
import json
from datetime import datetime, timezone

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

comparacao = {
    "escopo": "Um par de inspecao; nao representa o conjunto de teste",
    "suporte": nomes_par["suporte"],
    "query": nomes_par["query"],
    "ALP": metricas_oficiais,
    "ALP_SAM": metricas_sam,
    "delta_Dice_pontos_percentuais": 100 * (
        metricas_sam["Dice"] - metricas_oficiais["Dice"]
    ),
    "configuracao_prompts": {
        "use_cca": False,
        "point_mode": "both",
        "use_bbox": True,
        "use_neg_points": False,
        "selecao_mascara_sam": "Primeira candidata, conforme codigo oficial",
    },
    "evidencias": {
        "preparo": str(PREPARO_DIR),
        "ALP": str(PRED_ALP_DIR),
        "prompts_inspecao_CPU": str(PROMPTS_DIR),
        "pipeline_completo": str(PRED_SAM_DIR),
        "figura": str(PRED_SAM_DIR / "figuras/comparacao_alp_sam.png"),
    },
    "pendencias": [
        "Consolidar ambiente completo e registro de console",
        "Conferir a imagem efetivamente entregue ao predictor SAM",
        "Conferir argumentos completos do script oficial de avaliacao",
        "Definir e executar avaliacao completa, com suportes fixos",
        "Medir desempenho com protocolo proprio de benchmark",
        "Migrar a integracao VMamba em commits separados",
    ],
}

conclusao = f"""# Conclusão parcial — primeiro par oficial

O pipeline DINOv2-L + ALP + SAM-H executou com pesos locais.
O código-base é o commit oficial 64868c909668aac9ad517f7192a2712e88268e8f.
O carregamento do DINOv2 foi adaptado no notebook para usar arquivos locais.

ALP: Dice {metricas_oficiais['Dice']:.6f}.
ALP + SAM: Dice {metricas_sam['Dice']:.6f}.

Nesta query, o SAM reduziu falsos positivos e aumentou falsos negativos.
A figura mostra preenchimento de um buraco interno e maior omissão nas bordas.
O resultado não estabelece o efeito médio do SAM.

Avaliação contra a mesma máscara oficial pré-processada, sem padding.
A query é de inspeção, não do conjunto de teste.
Não houve ajuste de parâmetros em função do resultado.

O campo original_size do dataset contém dimensões incorretas.
Nesta execução, as dimensões de avaliação vieram da imagem real e do resize.

A reprodução quantitativa do artigo e a migração VMamba permanecem pendentes.
"""

destino_json = RUN_DIR / f"resumo_par_{identificador}.json"
destino_md = RUN_DIR / f"conclusao_par_{identificador}.md"

with destino_json.open("x", encoding="utf-8") as arquivo:
    json.dump(comparacao, arquivo, indent=2, ensure_ascii=False)
with destino_md.open("x", encoding="utf-8") as arquivo:
    arquivo.write(conclusao)

print(conclusao)
print(f"Resumo salvo: {destino_json}")
print(f"Conclusão salva: {destino_md}")

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image, display
from datetime import datetime, timezone
from models.ProtoSAM import ALPNetInput

captura = {}

class ImagemCapturada(Exception):
    pass

def capturar_imagem(image, *args, **kwargs):
    captura["imagem"] = image.copy()
    captura["formato"] = (
        args[0] if args else kwargs.get("image_format", "RGB")
    )
    raise ImagemCapturada()

# Guarda a configuração original do método para restaurá-la no finally.
predictor = protosam.predictor
tinha_override = "set_image" in predictor.__dict__
override_anterior = predictor.__dict__.get("set_image")
predictor.set_image = capturar_imagem

try:
    entrada_inspecao = ALPNetInput(
        support_images=[imagem_s],
        support_labels=[mascara_s],
        query_images=imagem_q,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        protosam(
            query_image=imagem_q,
            coarse_model_input=entrada_inspecao,
            degrees_rotate=0,
        )
except ImagemCapturada:
    print("Imagem capturada antes da execução do SAM.")
finally:
    if tinha_override:
        predictor.set_image = override_anterior
    else:
        del predictor.set_image

assert "imagem" in captura, "O pipeline não chegou a set_image."
imagem_sam = captura["imagem"]
assert imagem_sam.shape == (1024, 1024, 3)
assert imagem_sam.dtype == np.uint8
assert captura["formato"] == "RGB"

query = entradas_oficiais["query"]
h, w = map(int, query["tamanho_redimensionado"])
media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)

rgb_visualizacao = (
    (query["imagem"] * desvio + media)
    .clamp(0, 255).round().to(torch.uint8)
    .permute(1, 2, 0).numpy()
)

# Compara somente a região da imagem, excluindo o padding.
diferenca = np.abs(
    imagem_sam[:h, :w].astype(np.float32)
    - rgb_visualizacao[:h, :w].astype(np.float32)
)

diagnostico = {
    "origem": "Captura real na chamada a SamPredictor.set_image",
    "shape": list(imagem_sam.shape),
    "dtype": str(imagem_sam.dtype),
    "formato": captura["formato"],
    "comparacao": "RGB desnormalizado e arredondado; sem padding",
    "diferenca_absoluta_media": float(diferenca.mean()),
    "diferenca_absoluta_maxima": float(diferenca.max()),
    "media_por_canal_RGB": diferenca.mean(axis=(0, 1)).tolist(),
    "SAM_executado_nesta_inspecao": False,
}

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CAPTURA_DIR = SESSAO_DIR / f"entrada_sam_{identificador}"
CAPTURA_DIR.mkdir(exist_ok=False)
PILImage.fromarray(imagem_sam).save(CAPTURA_DIR / "entrada_real_sam.png")

with (CAPTURA_DIR / "diagnostico.json").open("x", encoding="utf-8") as arquivo:
    json.dump(diagnostico, arquivo, indent=2, ensure_ascii=False)

fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
eixos[0].imshow(rgb_visualizacao[:h, :w])
eixos[0].set_title("RGB usado nas visualizações")
eixos[1].imshow(imagem_sam[:h, :w])
eixos[1].set_title("Imagem efetivamente enviada ao SAM")
for eixo in eixos:
    eixo.axis("off")
fig.tight_layout()

destino = CAPTURA_DIR / "comparacao_entrada_sam.png"
fig.savefig(destino, dpi=150, bbox_inches="tight")
plt.close(fig)

print(json.dumps(diagnostico, indent=2, ensure_ascii=False))
display(Image(filename=str(destino)))
print(f"Salvo: {CAPTURA_DIR}")

In [ ]:
import ast
from datetime import datetime, timezone

partes = []

# Lê o script inteiro para incluir condições e variáveis em maiúsculas.
caminho = REPO_NOVO / "run_protosam.sh"
partes.append("--- run_protosam.sh completo ---")
for i, linha in enumerate(
    caminho.read_text(encoding="utf-8").splitlines(), start=1
):
    partes.append(f"{i:04d}: {linha}")

# Inspeciona seleção de suporte e avaliação, com contexto.
caminho = REPO_NOVO / "validation_protosam.py"
linhas = caminho.read_text(encoding="utf-8").splitlines()
termos = (
    "support_idx", "get_support", "support_images",
    "support_fg_mask", "DataLoader(",
    "record_prediction", "get_dice", "get_iou",
    "Dice", "dice", "Metric", "metric",
    "original_size", "original_sz", "ROT_DEG",
)

selecionadas = set()
for i, linha in enumerate(linhas):
    if any(termo in linha for termo in termos):
        selecionadas.update(
            range(max(0, i - 4), min(len(linhas), i + 5))
        )

partes.append("\n--- validation_protosam.py: suporte e avaliação ---")
anterior = -2
for i in sorted(selecionadas):
    if i > anterior + 1:
        partes.append("...")
    partes.append(f"{i + 1:04d}: {linhas[i]}")
    anterior = i

# Mostra os métodos do dataset responsáveis por selecionar suporte.
caminho = REPO_NOVO / "dataloaders/PolypDataset.py"
fonte = caminho.read_text(encoding="utf-8")
for classe in ast.parse(fonte).body:
    if isinstance(classe, ast.ClassDef) and classe.name == "PolypDataset":
        for metodo in classe.body:
            if isinstance(metodo, ast.FunctionDef) and "support" in metodo.name:
                partes.append(f"\n--- PolypDataset.{metodo.name} ---")
                partes.append(ast.get_source_segment(fonte, metodo))

texto = "\n".join(partes)
print(texto)

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"protocolo_avaliacao_{identificador}.txt"
with destino.open("x", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print(f"\nSalvo: {destino}")

In [ ]:
import ast
from datetime import datetime, timezone

partes = []

relativo = "config_ssl_upload.py"
linhas = (REPO_NOVO / relativo).read_text(encoding="utf-8").splitlines()
partes.append(f"--- {relativo} completo ---")
partes.extend(
    f"{i:04d}: {linha}"
    for i, linha in enumerate(linhas, start=1)
)

relativo = "validation_protosam.py"
fonte = (REPO_NOVO / relativo).read_text(encoding="utf-8")
arvore = ast.parse(fonte)

for no in arvore.body:
    if isinstance(no, ast.FunctionDef) and no.name in (
        "get_alpnet_model", "get_model", "main"
    ):
        partes.append(f"\n--- {relativo}: {no.name} ---")
        partes.append(ast.get_source_segment(fonte, no))

texto = "\n".join(partes)
print(texto)

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = SESSAO_DIR / f"config_e_avaliacao_completas_{identificador}.txt"
with destino.open("x", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print(f"\nSalvo: {destino}")

In [ ]:
import csv
import json
from pathlib import Path
from datetime import datetime, timezone

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AVALIACAO_DIR = (
    REPO_NOVO / "reproducao" / f"01_avaliacao_K1_{identificador}"
)
AVALIACAO_DIR.mkdir(parents=True, exist_ok=False)

assert len(ds_teste.images) == len(ds_teste.gts) == 100

manifesto = []
for indice, (imagem, mascara) in enumerate(
    zip(ds_teste.images, ds_teste.gts)
):
    imagem = Path(imagem).resolve()
    mascara = Path(mascara).resolve()
    assert imagem.is_file() and mascara.is_file()
    assert imagem.stem == mascara.stem
    assert imagem.name != nomes_par["suporte"]

    manifesto.append({
        "indice": indice,
        "query": imagem.name,
        "imagem": str(imagem),
        "mascara": str(mascara),
    })

protocolo = {
    "status": "Preparado; inferencia ainda nao executada",
    "base_oficial": "64868c909668aac9ad517f7192a2712e88268e8f",
    "tarefa": "Segmentacao binaria one-shot de polipos",
    "suporte": nomes_par["suporte"],
    "selecao_suporte": "K1 fixado previamente por nome; sem sorteio",
    "quantidade_queries_manifesto": len(manifesto),
    "dataset": "polyps",
    "config_alp": dict(alpnet.config),
    "resolucao_encoder": alpnet.image_size,
    "config_protosam": dict(config_protosam),
    "isval": True,
    "val_wsize": 2,
    "degrees_rotate": 0,
    "dtype": "float32",
    "atualizacao_de_pesos": False,
    "metrica_principal": "Media do Dice por imagem avaliada",
    "espaco_metricas": "Canvas completo 1024x1024, como na avaliacao oficial",
    "queries_sem_objeto": "Ignorar e registrar explicitamente no CSV",
    "predicao_vazia_com_objeto": "Dice, IoU, precisao e recall iguais a zero",
    "preprocessamento": "Oficial, incluindo conversao min-max antes do SAM",
    "tempos": "Registrar por imagem; nao tratar como benchmark controlado",
    "pendencias": [
        "Registrar ambiente efetivo e semente antes da execucao",
        "Implementar avaliacao e salvamento incremental",
    ],
}

with (AVALIACAO_DIR / "manifesto_teste.csv").open(
    "x", newline="", encoding="utf-8"
) as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=list(manifesto[0]))
    writer.writeheader()
    writer.writerows(manifesto)

with (AVALIACAO_DIR / "protocolo.json").open(
    "x", encoding="utf-8"
) as arquivo:
    json.dump(protocolo, arquivo, indent=2, ensure_ascii=False)

print("Suporte:", protocolo["suporte"])
print("Queries no manifesto:", len(manifesto))
print("Primeira:", manifesto[0]["query"])
print("Última:", manifesto[-1]["query"])
print("CCA:", protocolo["config_protosam"]["use_cca"])
print("Diretório:", AVALIACAO_DIR)

In [ ]:
import os
import sys
import json
import random
import platform
import subprocess
import importlib.metadata
import numpy as np
import torch
from datetime import datetime, timezone

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
torch.cuda.manual_seed_all(SEMENTE)

protosam.eval()
alpnet.eval()
assert not any(p.requires_grad for p in alpnet.parameters())
assert not any(p.requires_grad for p in protosam.sam.parameters())

# Confere os parâmetros do objeto realmente carregado.
for nome in (
    "use_cca", "point_mode", "use_bbox", "use_points",
    "use_mask", "use_neg_points", "coarse_pred_only",
    "num_points_for_sam",
):
    assert getattr(protosam, nome) == protocolo["config_protosam"][nome], nome

assert alpnet.image_size == protocolo["resolucao_encoder"]

def comando(args):
    resultado = subprocess.run(
        args, cwd=REPO_NOVO, capture_output=True, text=True
    )
    return resultado.stdout.strip() if resultado.returncode == 0 else "indisponivel"

cpu = platform.processor()
cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.is_file():
    cpu = next(
        (linha.split(":", 1)[1].strip()
         for linha in cpuinfo.read_text().splitlines()
         if linha.startswith("model name")),
        cpu,
    )

ambiente_avaliacao = {
    "data_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "executavel": sys.executable,
    "commit": comando(["git", "rev-parse", "HEAD"]),
    "diff_rastreado": comando(["git", "diff", "HEAD", "--stat"]),
    "cpu": cpu,
    "cpus_logicas": os.cpu_count(),
    "threads": torch.get_num_threads(),
    "threads_interop": torch.get_num_interop_threads(),
    "gpu_driver": comando([
        "nvidia-smi",
        "--query-gpu=name,uuid,driver_version,memory.total",
        "--format=csv,noheader",
    ]),
    "cuda_pytorch": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cudnn_enabled": torch.backends.cudnn.enabled,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "semente": SEMENTE,
    "bibliotecas": {
        d.metadata["Name"]: d.version
        for d in importlib.metadata.distributions()
        if d.metadata["Name"]
    },
}

config_avaliacao = {
    **protocolo,
    "status": "Pronto para executar",
    "semente": SEMENTE,
    "autocast": False,
    "modo": "eval + inference_mode",
    "cache_features_suporte": False,
    "carregamento_queries": "Sequencial no processo do notebook",
    "pendencias": ["Executar e salvar resultados incrementalmente"],
}

for nome, conteudo in (
    ("ambiente.json", ambiente_avaliacao),
    ("config.json", config_avaliacao),
):
    with (AVALIACAO_DIR / nome).open("x", encoding="utf-8") as arquivo:
        json.dump(conteudo, arquivo, indent=2, ensure_ascii=False)

with (AVALIACAO_DIR / "pergunta.md").open("x", encoding="utf-8") as arquivo:
    arquivo.write(
        "# Avaliação controlada K1\n\n"
        "Qual é o Dice médio por imagem do pipeline oficial DINOv2-L "
        "+ ALP + SAM-H nas queries do manifesto, com suporte K1 fixo?\n\n"
        "Manter os parâmetros de config.json durante toda a execução. "
        "Registrar exclusões, falhas, métricas e máscaras por imagem.\n\n"
        "Esta execução estabelece uma referência para a futura comparação "
        "com VMamba; não comprova reprodução de uma tabela do artigo.\n"
    )

print("Semente:", SEMENTE)
print("Threads:", ambiente_avaliacao["threads"])
print("cuDNN benchmark:", ambiente_avaliacao["cudnn_benchmark"])
print("GPU/driver:", ambiente_avaliacao["gpu_driver"])
print("Configuração conferida. Nenhuma query avaliada nesta célula.")
print("Registros:", AVALIACAO_DIR)

In [ ]:
import csv
import json
import sys
import time
import traceback
import numpy as np
import torch
from pathlib import Path
from contextlib import ExitStack, redirect_stdout, redirect_stderr
from PIL import Image as PILImage
from models.ProtoSAM import ALPNetInput

# Impede sobrescrita acidental. Uma retomada será tratada separadamente.
for nome in ("por_imagem.csv", "tempos.csv", "console.log", "mascaras"):
    assert not (AVALIACAO_DIR / nome).exists(), (
        f"{nome} já existe. Não reexecute esta célula sobre resultados anteriores."
    )

with (AVALIACAO_DIR / "manifesto_teste.csv").open(encoding="utf-8") as f:
    itens = list(csv.DictReader(f))

assert len(itens) == 100
for item in itens:
    i = int(item["indice"])
    assert Path(ds_teste.images[i]).resolve() == Path(item["imagem"])
    assert Path(ds_teste.gts[i]).resolve() == Path(item["mascara"])

# Recarrega o suporte fixo pelo dataset oficial.
indices = [
    i for i, caminho in enumerate(ds_suportes.images)
    if Path(caminho).name == protocolo["suporte"]
]
assert len(indices) == 1
suporte_eval = ds_suportes[indices[0]]
device = next(alpnet.parameters()).device
sup_img = suporte_eval["image"].unsqueeze(0).to(device)
sup_mask = suporte_eval["label"].unsqueeze(0).to(device)

pasta_mascaras = AVALIACAO_DIR / "mascaras"
pasta_mascaras.mkdir()
resultados = []
estado = "em_andamento"
erro = None

class Tee:
    def __init__(self, tela, arquivo):
        self.tela, self.arquivo = tela, arquivo
    def write(self, texto):
        self.tela.write(texto)
        self.arquivo.write(texto)
        self.flush()
        return len(texto)
    def flush(self):
        self.tela.flush()
        self.arquivo.flush()

campos = [
    "indice", "query", "suporte", "status", "TP", "FP", "FN",
    "Dice", "IoU", "precisao", "recall", "numero_scores_sam",
    "scores_sam", "imagem", "mascara_referencia", "predicao",
]

protosam.eval()
alpnet.eval()

with ExitStack() as stack:
    log = stack.enter_context(
        (AVALIACAO_DIR / "console.log").open("x", encoding="utf-8")
    )
    tabela = stack.enter_context(
        (AVALIACAO_DIR / "por_imagem.csv").open(
            "x", newline="", encoding="utf-8"
        )
    )
    tempos = stack.enter_context(
        (AVALIACAO_DIR / "tempos.csv").open(
            "x", newline="", encoding="utf-8"
        )
    )
    writer = csv.DictWriter(tabela, fieldnames=campos)
    writer.writeheader()
    timer = csv.writer(tempos)
    timer.writerow(["indice", "query", "segundos_forward"])
    tabela.flush()
    tempos.flush()

    stack.enter_context(redirect_stdout(Tee(sys.stdout, log)))
    stack.enter_context(redirect_stderr(Tee(sys.stderr, log)))

    try:
        with torch.inference_mode(), torch.autocast("cuda", enabled=False):
            for posicao, item in enumerate(itens, start=1):
                indice = int(item["indice"])
                exemplo = ds_teste[indice]
                gt = exemplo["label"].detach().cpu().bool()

                linha = {
                    "indice": indice,
                    "query": item["query"],
                    "suporte": protocolo["suporte"],
                    "imagem": item["imagem"],
                    "mascara_referencia": item["mascara"],
                }

                if not gt.any():
                    linha["status"] = "ignorada_sem_objeto"
                    writer.writerow(linha)
                    tabela.flush()
                    resultados.append(linha)
                    print(f"[{posicao}/100] {item['query']} — sem objeto; ignorada")
                    continue

                qry = exemplo["image"].unsqueeze(0).to(device)
                entrada = ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True,
                    val_wsize=2,
                    show_viz=False,
                    supp_fts=None,
                )

                torch.cuda.synchronize()
                inicio = time.perf_counter()
                pred, scores = protosam(
                    query_image=qry,
                    coarse_model_input=entrada,
                    degrees_rotate=0,
                )
                torch.cuda.synchronize()
                segundos = time.perf_counter() - inicio

                pred = pred.detach().cpu()
                assert pred.shape == gt.shape == (1024, 1024)
                assert torch.isfinite(pred).all()
                assert ((pred == 0) | (pred == 1)).all()
                pred = pred.bool()

                tp = int((pred & gt).sum().item())
                fp = int((pred & ~gt).sum().item())
                fn = int((~pred & gt).sum().item())

                destino = pasta_mascaras / f"{indice:04d}_{Path(item['query']).stem}.png"
                PILImage.fromarray(
                    pred.numpy().astype(np.uint8) * 255
                ).save(destino)

                linha.update({
                    "status": "avaliada",
                    "TP": tp, "FP": fp, "FN": fn,
                    "Dice": 2 * tp / (2 * tp + fp + fn + 1e-8),
                    "IoU": tp / (tp + fp + fn + 1e-8),
                    "precisao": tp / (tp + fp + 1e-8),
                    "recall": tp / (tp + fn + 1e-8),
                    "numero_scores_sam": len(scores),
                    "scores_sam": json.dumps([float(s) for s in scores]),
                    "predicao": str(destino),
                })
                timer.writerow([indice, item["query"], segundos])
                tempos.flush()
                writer.writerow(linha)
                tabela.flush()
                resultados.append(linha)

                print(
                    f"[{posicao}/100] Dice={linha['Dice']:.4f} "
                    f"forward={segundos:.3f}s — {item['query']}"
                )

        estado = "concluida"
    except BaseException as exc:
        estado = "interrompida"
        erro = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
        raise
    finally:
        validos = [r for r in resultados if r["status"] == "avaliada"]
        resumo = {
            "status": estado,
            "previstas": len(itens),
            "avaliadas": len(validos),
            "ignoradas_sem_objeto": len(resultados) - len(validos),
            "pendentes": len(itens) - len(resultados),
            "medias_por_imagem": {
                chave: float(np.mean([r[chave] for r in validos]))
                if validos else None
                for chave in ("Dice", "IoU", "precisao", "recall")
            },
            "erro": erro,
            "metricas": "Canvas completo; media por imagem avaliada",
            "mascaras": "PNG binario: fundo=0, objeto=255",
            "tempos": "Forward sincronizado; sem benchmark controlado",
        }
        with (AVALIACAO_DIR / "resumo.json").open(
            "x", encoding="utf-8"
        ) as f:
            json.dump(resumo, f, indent=2, ensure_ascii=False)
        print("\n", json.dumps(resumo, indent=2, ensure_ascii=False))

In [ ]:
import csv
import json
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# Caminho explícito permite consultar os resultados após reiniciar o kernel.
AVALIACAO_DIR = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "01_avaliacao_K1_20261004T140725337013Z"
)

with (AVALIACAO_DIR / "por_imagem.csv").open(encoding="utf-8") as f:
    linhas = list(csv.DictReader(f))
with (AVALIACAO_DIR / "resumo.json").open(encoding="utf-8") as f:
    resumo_salvo = json.load(f)

avaliadas = [r for r in linhas if r["status"] == "avaliada"]
assert len(linhas) == len(avaliadas) == 100
assert len({r["query"] for r in linhas}) == 100
assert all(Path(r["predicao"]).is_file() for r in avaliadas)

medias = {
    chave: float(np.mean([float(r[chave]) for r in avaliadas]))
    for chave in ("Dice", "IoU", "precisao", "recall")
}
for chave, valor in medias.items():
    assert np.isclose(
        valor, resumo_salvo["medias_por_imagem"][chave],
        rtol=0, atol=1e-12,
    ), f"Média divergente: {chave}"

ordenadas = sorted(avaliadas, key=lambda r: float(r["Dice"]))
piores = [
    {
        "query": r["query"],
        "Dice": float(r["Dice"]),
        "precisao": float(r["precisao"]),
        "recall": float(r["recall"]),
        "numero_scores_sam": int(r["numero_scores_sam"]),
    }
    for r in ordenadas[:5]
]

conclusao = f"""# Baseline controlado — suporte K1

Execução concluída: 100 imagens avaliadas, nenhuma excluída.
DINOv2-L, entrada 672, ALP e SAM-H.
Suporte K1 fixo; CCA desativada; pontos de confiança e centroide mais caixa.
Pré-processamento do modelo oficial, com carregamento local dos pesos.

Médias por imagem no canvas completo:
- Dice: {medias['Dice']:.6f}
- IoU: {medias['IoU']:.6f}
- Precisão: {medias['precisao']:.6f}
- Recall: {medias['recall']:.6f}

As médias foram conferidas contra por_imagem.csv.
Os 100 caminhos de máscaras salvas existem.

Limites:
- Um único suporte; não mede variabilidade entre suportes.
- Tempos registrados não constituem benchmark controlado.
- Não demonstra, isoladamente, reprodução dos resultados publicados.
- Não houve comparação agregada com ALP sem SAM nesta execução.

Próximo passo:
Inspecionar os casos de menor Dice para caracterizar falhas,
sem ajustar parâmetros usando o conjunto de teste.
"""

identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = AVALIACAO_DIR / f"conclusao_{identificador}.md"
with destino.open("x", encoding="utf-8") as f:
    f.write(conclusao)

with (AVALIACAO_DIR / f"piores_casos_{identificador}.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(piores, f, indent=2, ensure_ascii=False)

print(conclusao)
print("Cinco menores Dice:")
print(json.dumps(piores, indent=2, ensure_ascii=False))
print(f"\nSalvo: {destino}")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import cv2
import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import display, Image

avaliacao = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "01_avaliacao_K1_20261004T140725337013Z"
)
assert "ds_teste" in globals(), (
    "O dataset ds_teste precisa estar carregado com o pré-processamento oficial."
)

df = pd.read_csv(avaliacao / "por_imagem.csv")
piores = df.nsmallest(5, "Dice")
indices = {Path(p).name: i for i, p in enumerate(ds_teste.images)}
assert len(indices) == len(ds_teste.images), "Nomes repetidos no dataset."

fig, axes = plt.subplots(5, 3, figsize=(15, 23))

for linha, (_, registro) in enumerate(piores.iterrows()):
    amostra = ds_teste[indices[registro["query"]]]
    gt = amostra["label"].detach().cpu().numpy() > 0.5

    png = cv2.imread(str(registro["predicao"]), cv2.IMREAD_GRAYSCALE)
    assert png is not None, registro["predicao"]
    assert set(np.unique(png)).issubset({0, 255})
    pred = png > 0
    assert gt.shape == pred.shape

    tp = pred & gt
    fp = pred & ~gt
    fn = ~pred & gt
    for nome, pixels in (("TP", tp), ("FP", fp), ("FN", fn)):
        assert int(pixels.sum()) == int(registro[nome]), (
            f"{registro['query']}: divergência em {nome}"
        )

    # Apenas para exibição: desfaz a normalização oficial da imagem.
    imagem = amostra["image"].detach().cpu().numpy().transpose(1, 2, 0)
    media = np.array([123.675, 116.28, 103.53])
    desvio = np.array([58.395, 57.12, 57.375])
    rgb = np.clip(np.rint(imagem * desvio + media), 0, 255).astype(np.uint8)

    # Mostra o padding em preto, usando as dimensões reais da imagem.
    original = cv2.imread(str(registro["imagem"]))
    assert original is not None
    h, w = original.shape[:2]
    escala = 1024 / max(h, w)
    hr, wr = int(h * escala + 0.5), int(w * escala + 0.5)
    rgb[hr:, :] = 0
    rgb[:, wr:] = 0

    axes[linha, 0].imshow(rgb)
    axes[linha, 0].contour(gt, levels=[0.5], colors=["lime"], linewidths=1)
    axes[linha, 0].set_title(f"{registro['query']}\nReferência: verde", fontsize=9)

    axes[linha, 1].imshow(rgb)
    if pred.any() and not pred.all():
        axes[linha, 1].contour(
            pred, levels=[0.5], colors=["cyan"], linewidths=1
        )
    axes[linha, 1].set_title(
        f"Predição: ciano | Dice={registro['Dice']:.4f}\n"
        f"Precisão={registro['precisao']:.3f} | Recall={registro['recall']:.3f}"
    )

    erros = np.zeros((*gt.shape, 3), dtype=np.uint8)
    erros[tp] = [0, 190, 0]
    erros[fp] = [255, 60, 60]
    erros[fn] = [60, 120, 255]
    axes[linha, 2].imshow(erros)
    axes[linha, 2].set_title("TP: verde | FP: vermelho | FN: azul")

    for ax in axes[linha]:
        ax.axis("off")

fig.tight_layout()
destino = avaliacao / "figures"
destino.mkdir(exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
arquivo = destino / f"cinco_menores_dice_{stamp}.png"
fig.savefig(arquivo, dpi=140, bbox_inches="tight")
plt.close(fig)

print("TP, FP e FN conferidos nos cinco casos.")
print("Salvo:", arquivo)
display(Image(filename=str(arquivo)))

In [ ]:
import os
from pathlib import Path

repo = Path("/workspace/ProtoSAM-repro")
os.chdir(repo)

from dataloaders.PolypDataset import PolypDataset
from dataloaders.PolypTransforms import get_polyp_transform
from models.segment_anything.utils.transforms import ResizeLongestSide

_, transform_test = get_polyp_transform()

ds_teste = PolypDataset(
    root="./data/PolypDataset/TestDataset/test/images/",
    image_root="./data/PolypDataset/TestDataset/test/images/",
    gt_root="./data/PolypDataset/TestDataset/test/masks/",
    train=False,
    augmentations=transform_test,
    sam_trans=ResizeLongestSide(1024),
    image_size=(1024, 1024),
)

assert len(ds_teste) == 100, f"Quantidade inesperada: {len(ds_teste)}"
print("Dataset de teste preparado:", len(ds_teste), "imagens.")

In [ ]:
for nome in ("alpnet", "protosam", "ds_teste"):
    obj = globals().get(nome)
    if obj is None:
        print(f"{nome}: não carregado")
    else:
        classe = type(obj)
        print(f"{nome}: {classe.__module__}.{classe.__name__}")

In [ ]:
from pathlib import Path
import inspect
import torch
import models.grid_proto_fewshot as modulo_alp

repo = Path("/workspace/ProtoSAM-repro")
arquivo_modulo = Path(inspect.getfile(modulo_alp)).resolve()
assert arquivo_modulo == repo / "models/grid_proto_fewshot.py", (
    f"Módulo carregado de outro repositório: {arquivo_modulo}"
)
assert torch.cuda.is_available(), "CUDA indisponível."

dino_repo = (
    repo / "pretrained_model/hub/facebookresearch_dinov2_main"
).resolve()
dino_pesos = (
    repo / "pretrained_model/hub/checkpoints/dinov2_vitl14_pretrain.pth"
).resolve()

assert (dino_repo / "hubconf.py").is_file(), dino_repo
assert dino_pesos.is_file(), dino_pesos

class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(dino_repo),
            "dinov2_vitl14",
            source="local",
            pretrained=False,
        )
        pesos = torch.load(
            dino_pesos, map_location="cpu", weights_only=True
        )
        print("Carregamento:", self.encoder.load_state_dict(pesos, strict=True))

        lado = max(
            self.image_size // 14,
            modulo_alp.DEFAULT_FEATURE_SIZE,
        )
        self.config["feature_hw"] = [lado, lado]

config_alp = {
    "which_model": "dinov2_l14",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False,
}

alpnet = FewShotSegDinoLocal(
    image_size=672,
    pretrained_path=None,
    cfg=config_alp.copy(),
).to(device="cuda", dtype=torch.float32)

alpnet.eval()
alpnet.requires_grad_(False)

print("Módulo:", arquivo_modulo)
print("Pesos:", dino_pesos)
print("Grade de características:", alpnet.config["feature_hw"])
print("Modo treino:", alpnet.training)
print("Parâmetros treináveis:", sum(
    p.numel() for p in alpnet.parameters() if p.requires_grad
))
print("DINOv2 + ALP preparados. Nenhuma query avaliada.")

In [ ]:
from pathlib import Path
import pandas as pd
from dataloaders.PolypDataset import PolypDataset
from dataloaders.PolypTransforms import get_polyp_transform
from models.segment_anything.utils.transforms import ResizeLongestSide

avaliacao = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "01_avaliacao_K1_20261004T140725337013Z"
)

resultados = pd.read_csv(avaliacao / "por_imagem.csv")
casos_diagnostico = resultados.nsmallest(5, "Dice").copy()

nome_suporte = "cju3xtufwiv9c0818djsc4cqd.png"
assert resultados["suporte"].eq(nome_suporte).all()

_, transform_test = get_polyp_transform()
ds_suportes = PolypDataset(
    root="./data/PolypDataset/TrainDataset/images/",
    image_root="./data/PolypDataset/TrainDataset/images/",
    gt_root="./data/PolypDataset/TrainDataset/masks/",
    train=True,
    augmentations=transform_test,
    sam_trans=ResizeLongestSide(1024),
    image_size=(1024, 1024),
)

encontrados = [
    i for i, caminho in enumerate(ds_suportes.images)
    if Path(caminho).name == nome_suporte
]
assert len(encontrados) == 1, encontrados
indice_suporte = encontrados[0]
assert Path(ds_suportes.gts[indice_suporte]).stem == Path(nome_suporte).stem

suporte_diagnostico = ds_suportes[indice_suporte]
assert tuple(suporte_diagnostico["image"].shape) == (3, 1024, 1024)
assert tuple(suporte_diagnostico["label"].shape) == (1024, 1024)

pixels_suporte = int((suporte_diagnostico["label"] > 0.5).sum())
assert pixels_suporte == 51940, (
    f"Máscara do suporte diferente do baseline: {pixels_suporte} pixels."
)

indices_teste = {
    Path(caminho).name: i
    for i, caminho in enumerate(ds_teste.images)
}
assert len(indices_teste) == len(ds_teste.images)
assert all(nome in indices_teste for nome in casos_diagnostico["query"])

print("Suporte:", nome_suporte)
print("Índice:", indice_suporte)
print("Pixels de primeiro plano:", pixels_suporte)
print("\nCasos selecionados:")
print(casos_diagnostico[["query", "Dice"]].to_string(index=False))
print("\nEntradas preparadas. Nenhuma inferência executada.")

In [ ]:
import json
from datetime import datetime, timezone
import cv2
import numpy as np
import pandas as pd
import torch
from models.ProtoSAM import ALPNetInput, ALPNetWrapper

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
diagnostico_dir = avaliacao / f"diagnostico_alp_sam_{stamp}"
diagnostico_dir.mkdir(exist_ok=False)

def metricas_binarias(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    eps = 1e-8
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "Dice": 2 * tp / (2 * tp + fp + fn + eps),
        "precisao": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }

device = next(alpnet.parameters()).device
sup_img = suporte_diagnostico["image"].unsqueeze(0).to(device)
sup_mask = suporte_diagnostico["label"].unsqueeze(0).to(device)
wrapper_alp = ALPNetWrapper(alpnet)
alpnet.eval()

protocolo = {
    "objetivo": "Diagnóstico dos cinco menores Dice finais",
    "avaliacao_origem": str(avaliacao),
    "suporte": nome_suporte,
    "queries": casos_diagnostico["query"].tolist(),
    "config_alp": alpnet.config,
    "image_size": 672,
    "isval": True,
    "val_wsize": 2,
    "dtype": str(next(alpnet.parameters()).dtype),
    "sam": "Predições salvas da avaliação original; não reexecutado",
}
(diagnostico_dir / "protocolo.json").write_text(
    json.dumps(protocolo, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

comparacoes = []

with torch.inference_mode():
    for _, registro in casos_diagnostico.iterrows():
        nome = registro["query"]
        amostra = ds_teste[indices_teste[nome]]
        gt = amostra["label"].cpu().numpy() > 0.5

        sam_png = cv2.imread(
            str(registro["predicao"]), cv2.IMREAD_GRAYSCALE
        )
        assert sam_png is not None, registro["predicao"]
        assert set(np.unique(sam_png)).issubset({0, 255})
        pred_sam = sam_png > 0
        assert pred_sam.shape == gt.shape

        met_sam = metricas_binarias(pred_sam, gt)
        for chave in ("TP", "FP", "FN"):
            assert met_sam[chave] == int(registro[chave]), (
                f"{nome}: referência divergente em {chave}"
            )

        entrada = ALPNetInput(
            support_images=[sup_img],
            support_labels=[sup_mask],
            query_images=amostra["image"].unsqueeze(0).to(device),
            isval=True,
            val_wsize=2,
        )
        escores = wrapper_alp(entrada)
        assert tuple(escores.shape) == (1, 2, 1024, 1024)
        assert torch.isfinite(escores).all()

        pred_alp = escores.argmax(dim=1)[0].cpu().numpy() > 0
        met_alp = metricas_binarias(pred_alp, gt)

        arquivo = diagnostico_dir / f"{Path(nome).stem}_alp.pt"
        torch.save({
            "query": nome,
            "suporte": nome_suporte,
            "escores": escores.cpu(),
            "predicao": torch.from_numpy(pred_alp.astype(np.uint8)),
        }, arquivo)

        linha = {
            "query": nome,
            "arquivo_alp": str(arquivo),
            "arquivo_sam": registro["predicao"],
            **{f"ALP_{k}": v for k, v in met_alp.items()},
            **{f"SAM_{k}": v for k, v in met_sam.items()},
            "delta_Dice_SAM_menos_ALP": met_sam["Dice"] - met_alp["Dice"],
        }
        comparacoes.append(linha)
        pd.DataFrame(comparacoes).to_csv(
            diagnostico_dir / "comparacao.csv", index=False
        )

        print(
            f"{nome}\n"
            f"  ALP: Dice={met_alp['Dice']:.4f}, "
            f"precisão={met_alp['precisao']:.4f}, "
            f"recall={met_alp['recall']:.4f}\n"
            f"  SAM: Dice={met_sam['Dice']:.4f}, "
            f"precisão={met_sam['precisao']:.4f}, "
            f"recall={met_sam['recall']:.4f}\n"
            f"  Δ Dice: {linha['delta_Dice_SAM_menos_ALP']:+.4f}",
            flush=True,
        )

print("\nSalvo:", diagnostico_dir)


In [ ]:
import inspect
from models.ProtoSAM import ProtoSAM
from util.utils import get_connected_components

for nome in (
    "get_bbox_per_cc",
    "get_sam_input_points",
    "get_most_conf_points",
):
    print(f"\n--- ProtoSAM.{nome} ---")
    print(inspect.getsource(getattr(ProtoSAM, nome)))

print("\n--- get_connected_components ---")
print(inspect.getsource(get_connected_components))


In [ ]:
import json
from pathlib import Path
from types import SimpleNamespace, MethodType
import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Image
from models.ProtoSAM import ProtoSAM
from util.utils import get_connected_components

diagnostico_dir = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "01_avaliacao_K1_20261004T140725337013Z/"
    "diagnostico_alp_sam_20261006T082641669776Z"
)
comparacao = pd.read_csv(diagnostico_dir / "comparacao.csv")
historico = pd.read_csv(diagnostico_dir.parent / "por_imagem.csv").set_index("query")

# Apenas os atributos e métodos necessários para gerar prompts.
extrator = SimpleNamespace(
    point_mode="both", num_points_for_sam=1, debug=False
)
extrator.get_most_conf_points = MethodType(
    ProtoSAM.get_most_conf_points, extrator
)

fig, axes = plt.subplots(len(comparacao), 4, figsize=(20, 23))
registros_prompts = []

for linha, registro in comparacao.iterrows():
    nome = registro["query"]
    salvo = torch.load(
        registro["arquivo_alp"], map_location="cpu", weights_only=True
    )
    logits = salvo["escores"]
    pred_alp = logits.argmax(1)[0].numpy().astype(np.uint8)
    componentes, _ = get_connected_components(pred_alp, logits)
    n_componentes = componentes[0] - 1

    if n_componentes:
        caixas = ProtoSAM.get_bbox_per_cc(extrator, componentes)
        pontos, _, _, _ = ProtoSAM.get_sam_input_points(
            extrator, componentes, logits.softmax(1),
            get_neg_points=False, l=1,
        )
    else:
        caixas, pontos = [], []

    amostra = ds_teste[indices_teste[nome]]
    gt = amostra["label"].cpu().numpy() > 0.5
    sam_png = cv2.imread(registro["arquivo_sam"], cv2.IMREAD_GRAYSCALE)
    assert sam_png is not None
    pred_sam = sam_png > 0
    assert pred_alp.shape == gt.shape == pred_sam.shape

    imagem = amostra["image"].cpu().numpy().transpose(1, 2, 0)
    rgb = np.clip(np.rint(
        imagem * np.array([58.395, 57.12, 57.375])
        + np.array([123.675, 116.28, 103.53])
    ), 0, 255).astype(np.uint8)

    original = cv2.imread(str(historico.loc[nome, "imagem"]))
    assert original is not None
    h, w = original.shape[:2]
    escala = 1024 / max(h, w)
    hr, wr = int(h * escala + 0.5), int(w * escala + 0.5)
    rgb[hr:, :] = 0
    rgb[:, wr:] = 0

    for ax in axes[linha]:
        ax.imshow(rgb)
        ax.axis("off")

    for coluna, mascara, cor in (
        (0, gt, "lime"),
        (1, pred_alp, "cyan"),
        (3, pred_sam, "cyan"),
    ):
        if mascara.any() and not mascara.all():
            axes[linha, coluna].contour(
                mascara, levels=[0.5], colors=[cor], linewidths=1
            )

    axes[linha, 0].set_title(f"{nome}\nReferência", fontsize=9)
    axes[linha, 1].set_title(f"ALP — Dice {registro['ALP_Dice']:.4f}")
    axes[linha, 2].set_title(
        f"{n_componentes} componentes\n"
        "Caixa: amarelo | confiança: × vermelho | centroide: + azul",
        fontsize=9,
    )
    axes[linha, 3].set_title(f"SAM — Dice {registro['SAM_Dice']:.4f}")

    detalhes = []
    for cc_id, (caixa, pts) in enumerate(zip(caixas, pontos), start=1):
        x1, y1, x2, y2 = map(float, caixa)
        axes[linha, 2].add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="yellow", linewidth=0.7,
        ))
        axes[linha, 2].scatter(
            pts[0, 0], pts[0, 1], marker="x", c="red", s=24
        )
        axes[linha, 2].scatter(
            pts[1, 0], pts[1, 1], marker="+", c="deepskyblue", s=30
        )
        detalhes.append({
            "componente": cc_id,
            "area_pixels": int(componentes[2][cc_id, cv2.CC_STAT_AREA]),
            "bbox_xyxy": [x1, y1, x2, y2],
            "pontos_xy": np.asarray(pts).tolist(),
            "ordem": ["maior_confianca", "centroide"],
        })

    n_sam = int(historico.loc[nome, "numero_scores_sam"])
    print(
        f"{nome}: componentes ALP={n_componentes}, "
        f"resultados SAM anteriores={n_sam}"
    )
    registros_prompts.append({
        "query": nome,
        "origem": "Reconstrução a partir do ALP recalculado",
        "numero_componentes": n_componentes,
        "numero_scores_sam_anterior": n_sam,
        "componentes": detalhes,
    })

fig.tight_layout()
arquivo_figura = diagnostico_dir / "referencia_alp_prompts_sam.png"
fig.savefig(arquivo_figura, dpi=140, bbox_inches="tight")
plt.close(fig)

(diagnostico_dir / "prompts_reconstruidos.json").write_text(
    json.dumps(registros_prompts, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nSalvo:", arquivo_figura)
display(Image(filename=str(arquivo_figura)))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import cv2
import numpy as np
import pandas as pd
import torch
from models.ProtoSAM import ALPNetInput, ALPNetWrapper

avaliacao = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "01_avaliacao_K1_20261004T140725337013Z"
)
diagnostico = avaliacao / "diagnostico_alp_sam_20261006T082641669776Z"

assert "alpnet" in globals() and "suporte_diagnostico" in globals()
assert not alpnet.training
assert not any(p.requires_grad for p in alpnet.parameters())

base = pd.read_csv(avaliacao / "por_imagem.csv")
assert len(base) == 100 and base["query"].is_unique
assert base["suporte"].eq("cju3xtufwiv9c0818djsc4cqd.png").all()

cache = pd.read_csv(diagnostico / "comparacao.csv").set_index("query")
assert len(cache) == 5 and cache.index.is_unique

indices = {Path(p).name: i for i, p in enumerate(ds_teste.images)}
assert len(indices) == len(ds_teste.images)
assert set(base["query"]).issubset(indices)

# Mantém o destino ao reexecutar esta célula no mesmo kernel.
if "avaliacao_alp_dir" not in globals():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    avaliacao_alp_dir = avaliacao / f"comparacao_completa_alp_sam_{stamp}"
    avaliacao_alp_dir.mkdir(exist_ok=False)

saida = Path(avaliacao_alp_dir)
(saida / "mascaras_alp").mkdir(exist_ok=True)
csv_saida = saida / "por_imagem.csv"

def medir(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    eps = 1e-8
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "Dice": 2 * tp / (2 * tp + fp + fn + eps),
        "IoU": tp / (tp + fp + fn + eps),
        "precisao": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }

def salvar_json(nome, conteudo):
    temporario = saida / f"{nome}.tmp"
    temporario.write_text(
        json.dumps(conteudo, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    temporario.replace(saida / nome)

linhas = pd.read_csv(csv_saida).to_dict("records") if csv_saida.exists() else []
concluidas = {r["query"] for r in linhas}
assert len(concluidas) == len(linhas)
assert concluidas.issubset(set(base["query"]))
for r in linhas:
    assert Path(r["predicao_alp"]).is_file()

device = next(alpnet.parameters()).device
sup_img = suporte_diagnostico["image"].unsqueeze(0).to(device)
sup_mask = suporte_diagnostico["label"].unsqueeze(0).to(device)
wrapper = ALPNetWrapper(alpnet)

if not (saida / "protocolo.json").exists():
    salvar_json("protocolo.json", {
        "avaliacao_sam": str(avaliacao),
        "cache_cinco_casos": str(diagnostico),
        "suporte": base["suporte"].iloc[0],
        "queries": base["query"].tolist(),
        "config_alp": alpnet.config,
        "image_size": alpnet.image_size,
        "isval": True,
        "val_wsize": 2,
        "dtype": str(next(alpnet.parameters()).dtype),
        "torch": str(torch.__version__),
        "threads": torch.get_num_threads(),
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "metricas": "Canvas completo; média por imagem",
        "sam": "Máscaras da avaliação anterior; sem nova inferência",
        "mascaras_alp": "PNG binário: fundo=0, objeto=255",
    })

print("Diretório:", saida)
print(f"Já concluídas: {len(concluidas)}/100", flush=True)

try:
    with torch.inference_mode():
        for posicao, (_, registro) in enumerate(base.iterrows(), start=1):
            nome = registro["query"]
            if nome in concluidas:
                continue

            amostra = ds_teste[indices[nome]]
            gt = amostra["label"].cpu().numpy() > 0.5
            assert gt.any(), f"Referência vazia inesperada: {nome}"

            sam_png = cv2.imread(
                str(registro["predicao"]), cv2.IMREAD_GRAYSCALE
            )
            assert sam_png is not None
            assert set(np.unique(sam_png)).issubset({0, 255})
            pred_sam = sam_png > 0
            assert pred_sam.shape == gt.shape

            met_sam = medir(pred_sam, gt)
            for chave in ("TP", "FP", "FN"):
                assert met_sam[chave] == int(registro[chave]), (
                    f"{nome}: divergência da referência em {chave}"
                )

            if nome in cache.index:
                salvo = torch.load(
                    cache.loc[nome, "arquivo_alp"],
                    map_location="cpu",
                    weights_only=True,
                )
                assert salvo["query"] == nome
                assert salvo["suporte"] == registro["suporte"]
                pred_alp = salvo["predicao"].numpy() > 0
                origem = "reutilizado"
            else:
                entrada = ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=amostra["image"].unsqueeze(0).to(device),
                    isval=True,
                    val_wsize=2,
                )
                logits = wrapper(entrada)
                assert tuple(logits.shape) == (1, 2, 1024, 1024)
                assert torch.isfinite(logits).all()
                pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
                origem = "nova_inferencia"
                del logits, entrada

            assert pred_alp.shape == gt.shape
            met_alp = medir(pred_alp, gt)

            if origem == "reutilizado":
                for chave in ("TP", "FP", "FN"):
                    assert met_alp[chave] == int(cache.loc[nome, f"ALP_{chave}"])

            caminho_pred = saida / "mascaras_alp" / nome
            assert cv2.imwrite(
                str(caminho_pred), pred_alp.astype(np.uint8) * 255
            )

            delta = met_sam["Dice"] - met_alp["Dice"]
            linhas.append({
                "query": nome,
                "suporte": registro["suporte"],
                "origem_alp": origem,
                "predicao_alp": str(caminho_pred),
                "predicao_sam": registro["predicao"],
                **{f"ALP_{k}": v for k, v in met_alp.items()},
                **{f"SAM_{k}": v for k, v in met_sam.items()},
                "delta_Dice_SAM_menos_ALP": delta,
            })

            temporario = saida / "por_imagem.tmp.csv"
            pd.DataFrame(linhas).to_csv(temporario, index=False)
            temporario.replace(csv_saida)
            concluidas.add(nome)

            print(
                f"[{posicao}/100] ALP={met_alp['Dice']:.4f} "
                f"SAM={met_sam['Dice']:.4f} Δ={delta:+.4f} "
                f"({origem}) — {nome}",
                flush=True,
            )

except BaseException as erro:
    salvar_json("status.json", {
        "status": "interrompida",
        "concluidas": len(concluidas),
        "erro": repr(erro),
    })
    raise

comparacao_completa = pd.DataFrame(linhas)
assert len(comparacao_completa) == 100
delta = comparacao_completa["delta_Dice_SAM_menos_ALP"]
tolerancia = 1e-6
metricas = ("Dice", "IoU", "precisao", "recall")

resumo = {
    "status": "concluida",
    "avaliadas": 100,
    "medias_por_imagem": {
        modelo: {
            m: float(comparacao_completa[f"{modelo}_{m}"].mean())
            for m in metricas
        }
        for modelo in ("ALP", "SAM")
    },
    "delta_Dice_medio_SAM_menos_ALP": float(delta.mean()),
    "delta_Dice_mediano_SAM_menos_ALP": float(delta.median()),
    "SAM_melhora": int((delta > tolerancia).sum()),
    "SAM_piora": int((delta < -tolerancia).sum()),
    "empates": int((delta.abs() <= tolerancia).sum()),
    "tolerancia_empate": tolerancia,
    "ALP_reutilizados": int(
        comparacao_completa["origem_alp"].eq("reutilizado").sum()
    ),
}
salvar_json("resumo.json", resumo)
salvar_json("status.json", {"status": "concluida", "concluidas": 100})

print("\n", json.dumps(resumo, indent=2, ensure_ascii=False))
print("\nSalvo:", saida)

In [ ]:
from pathlib import Path

repo = Path("/workspace/ProtoSAM-repro")
pasta = repo / "exp"

print("Notebooks existentes:")
if pasta.exists():
    for arquivo in sorted(pasta.rglob("*.ipynb")):
        if ".ipynb_checkpoints" not in arquivo.parts:
            print(arquivo.relative_to(repo))
else:
    print("A pasta exp ainda não existe.")